# Dalmia Cement Data Platform
### Problem Statement
You need to build a data platform for Dalmia Cement that helps the company track its daily operations and make better business decisions.

### 🔹 What is the problem?
The company receives a large amount of data every day, such as:
* Orders (how much cement is ordered)
* Dispatches (how much is delivered)
* Inventory (how much stock is left)
* Dealer sales (how much dealers sell)

### 🔹 Data Lake Architecture
1. **Bronze Layer**: Raw data collection from many places
2. **Silver Layer**: cleaned and transformed data (standardized, no duplicates,and many things as per need).
3. **Gold Layer**: Business-level aggregations and KPIs for decision making.

### Step 1: Install Necessary Dependencies

In [227]:
!pip install delta-spark==3.3.0 -q

### Step 2: Initialize Spark Session with Delta Lake Support

In [228]:
import os
import sys
import platform
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, lower
import pyspark.sql.functions as F

if platform.system() == "Windows":
    # Fix for Windows: Set JAVA_HOME programmatically if running locally
    # Note: Adjust this path if your JDK 11 is installed elsewhere
    if "JAVA_HOME" not in os.environ:
        os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-11"
    print("Windows detected. JAVA_HOME set to:", os.environ["JAVA_HOME"])
else:
    print("Linux/Colab detected. Using default Java environment.")

builder = SparkSession.builder \
    .appName("DalmiaCement") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.warehouse.dir", "/content/spark-warehouse")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark version:", spark.version)
print("Session ready!")

Linux/Colab detected. Using default Java environment.
Spark version: 3.5.8
Session ready!


## 1. Bronze Layer: Raw Data Ingestion

### 1.1 Creating Orders Data (How much cement is ordered)

In [229]:
orders_data = [
    (1, "2026-03-01", 101, 1001, 50, 300, "Delivered", "North"),
    (2, "2026-03-02", 102, 1002, 30, 320, "completed", "West"),
    (3, "2026-03-03", 103, 1001, 20, 300, "cancelled", "South"),
    (4, "2026-03-04", 101, 1003, 40, 350, "completed", "East"),
    (5, "2026-03-05", 104, 1002, 60, 310, "Placed", "North"),
    (6, "2026-03-06", 105, 1004, None, 330, "completed", "West"),
    (7, "2026-03-07", 106, 1005, 25, None, "Delivered", "South"),
    (8, "2026-03-08", 101, 1001, 50, 300, "delivered", "North"),
    (9, "2026-03-09", 107, 1006, -10, 340, "completed", "East"),
    (10, "2026-03-10", 108, 1007, 45, 360, "CANCELLED", None)
]
orders_cols = ["order_id", "order_date", "dealer_id", "product_id", "quantity", "price", "order_status", "region"]
orders_df = spark.createDataFrame(orders_data, orders_cols)

### 1.2 Creating Dispatches Data (How much is delivered)

In [230]:
dispatch_data = [
    (201, 1, "2026-03-02", 101, 1001, 50, "shipped"),
    (202, 2, "2026-03-03", 102, 1002, 30, "not_delivered"),
    (203, None, "2026-03-05", 101, 1003, 40, "delivered"),
    (204, 3, "2026-03-06", 103, 1001, 20, "shipped"),
    (205, 4, "2026-03-07", 101, 1003, 35, "delivered"),
    (206, None, "2026-03-08", 104, 1002, 25, "pending"),
    (207, 5, "2026-03-09", 105, 1004, 60, "shipped"),
    (208, 6, "2026-03-10", 106, 1005, 45, "delivered"),
    (209, None, "2026-03-11", 102, 1002, 15, "cancelled"),
    (210, 7, "2026-03-12", 103, 1001, 55, "delivered")
]
dispatch_cols = ["dispatch_id", "order_id", "dispatch_date", "dealer_id", "product_id", "quantity_dispatched", "status"]
dispatch_df = spark.createDataFrame(dispatch_data, dispatch_cols)

### 1.3 Creating Inventory Data (How much stock is left)

In [231]:
inventory_data = [
    (1001, "W1", 500, "2026-03-05", 100),
    (1002, "W1", 300, "2026-03-05", 80),
    (1003, "W2", 200, "2026-03-05", 50),
    (1004, "W2", 45, "2026-03-05", 60)
]
inventory_cols = ["product_id", "warehouse_id", "stock_quantity", "last_updated", "reorder_level"]
inventory_df = spark.createDataFrame(inventory_data, inventory_cols)

### 1.4 Creating Dealer Sales Data (How much dealers sell)

In [232]:
sales_data = [
    (101, 1001, "2026-03-06", 45, 13500, "North"),
    (102, 1002, "2026-03-06", 25, 8000, "West"),
    (103, 1001, "2026-03-07", 15, 4500, "South"),
    (101, 1003, "2026-03-07", 35, 12250, "East")
]
sales_cols = ["dealer_id", "product_id", "sale_date", "quantity_sold", "revenue", "region"]
sales_df = spark.createDataFrame(sales_data, sales_cols)

###1.5 Creating Routes Data

In [233]:
data = [
    (1, "A", 30, 5.0),
    (2, "A", 35, 5.5),
    (3, "A", 28, 4.8),

    (4, "B", 40, 6.5),
    (5, "B", 42, 6.8),
    (6, "B", 38, 6.2),

    (7, "C", 25, 4.0),
    (8, "C", 27, 4.2),
    (9, "C", 26, 4.1)
]

columns = ["trip_id", "route", "time_taken_min", "fuel_used_litre"]

routes_df = spark.createDataFrame(data, columns)
routes_df.show()

+-------+-----+--------------+---------------+
|trip_id|route|time_taken_min|fuel_used_litre|
+-------+-----+--------------+---------------+
|      1|    A|            30|            5.0|
|      2|    A|            35|            5.5|
|      3|    A|            28|            4.8|
|      4|    B|            40|            6.5|
|      5|    B|            42|            6.8|
|      6|    B|            38|            6.2|
|      7|    C|            25|            4.0|
|      8|    C|            27|            4.2|
|      9|    C|            26|            4.1|
+-------+-----+--------------+---------------+



##New data coming (as example day-2)

In [264]:
new_orders = spark.createDataFrame([
    (2,  "2026-03-02", 102, 1002, 35, 320, "completed", "North_West"),
    (11, "2026-03-11", 109, 1008, 55, 380, "placed",    "North_east"),
], ["order_id","order_date","dealer_id","product_id","quantity","price","order_status","region"])

### 1.6 Saving Raw Data to Bronze Layer (Parquet format)

In [234]:
orders_df.write.mode("overwrite").parquet("data/bronze/orders")
dispatch_df.write.mode("overwrite").parquet("data/bronze/dispatch")
inventory_df.write.mode("overwrite").parquet("data/bronze/inventory")
sales_df.write.mode("overwrite").parquet("data/bronze/sales")
print(" Bronze data saved!")

 Bronze data saved!


## 2. Silver Layer: Data Cleaning

### 2.1 Cleaning Orders Data

#### 2.1.1 Reading Orders Raw Data

In [235]:
silver_orders = spark.read.parquet("data/bronze/orders")

#### 2.1.2 Casting Date Column

In [236]:
silver_orders = silver_orders.withColumn("order_date", col("order_date").cast("date"))
silver_orders.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- order_date: date (nullable = true)
 |-- dealer_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- quantity: long (nullable = true)
 |-- price: long (nullable = true)
 |-- order_status: string (nullable = true)
 |-- region: string (nullable = true)



#### 2.1.3 Removing Duplicate Orders

In [237]:
silver_orders = silver_orders.dropDuplicates(["order_id"])

#### 2.1.4 Filling Null Values

In [238]:
silver_orders = silver_orders.fillna({"quantity": 0, "price": 0, "region": "unknown"})

#### 2.1.5 Filtering Invalid Quantities

In [239]:
silver_orders = silver_orders.filter(col("quantity") > 0)

#### 2.1.6 Standardizing Order Status (Lowercase)

In [240]:
silver_orders = silver_orders.withColumn("order_status", lower(col("order_status")))

#### 2.1.7 Filtering Valid Statuses

In [241]:
silver_orders = silver_orders.filter(col("order_status").isin("placed", "completed", "cancelled", "delivered"))


### 2.2 Cleaning Dispatch Data

#### 2.2.1 Reading Dispatch Raw Data

In [242]:
silver_dispatch = spark.read.parquet("data/bronze/dispatch")

#### 2.2.2 Casting Dispatch Date

In [243]:
silver_dispatch = silver_dispatch.withColumn("dispatch_date", col("dispatch_date").cast("date"))

#### 2.2.3 Removing Duplicates

In [244]:
silver_dispatch = silver_dispatch.dropDuplicates(["dispatch_id"])

#### 2.2.4 Handling Null IDs

In [245]:
silver_dispatch = silver_dispatch.fillna({"order_id": 0, "dealer_id": 0})

#### 2.2.5 Filtering Quantity Dispatched

In [246]:
silver_dispatch = silver_dispatch.filter(col("quantity_dispatched") > 0)

#### 2.2.6 Standardizing Dispatch Status

In [247]:
silver_dispatch = silver_dispatch.withColumn("status", lower(col("status")))

#### 2.2.7 Filtering Valid Dispatch Statuses

In [248]:
silver_dispatch = silver_dispatch.filter(col("status").isin("shipped", "delivered", "pending", "cancelled", "not_delivered"))

### 2.3 Cleaning Inventory Data

#### 2.3.1 Reading Inventory Data

In [249]:
silver_inventory = spark.read.parquet("data/bronze/inventory")

#### 2.3.2 Casting Last Updated Date

In [250]:
silver_inventory = silver_inventory.withColumn("last_updated", col("last_updated").cast("date"))

### 2.4 Cleaning Sales Data

#### 2.4.1 Reading Sales Data

In [251]:
silver_sales = spark.read.parquet("data/bronze/sales")

#### 2.4.2 Casting Sale Date

In [252]:
silver_sales = silver_sales.withColumn("sale_date", col("sale_date").cast("date"))

### 2.5 Saving Cleaned Data to Silver Layer (Delta Tables)

##why we need delta table
We use Delta format to enable UPSERT operations, ensure data consistency using ACID transactions, and maintain versioned, reliable datasets

In [253]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

# Define paths for Delta tables
orders_path = "/content/spark-warehouse/silver/orders"
dispatch_path = "/content/spark-warehouse/silver/dispatch"
inventory_path = "/content/spark-warehouse/silver/inventory"
sales_path = "/content/spark-warehouse/silver/sales"

# Save Orders data as Delta and register table
silver_orders.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(orders_path)
spark.sql("DROP TABLE IF EXISTS silver.orders")
spark.sql(f"CREATE TABLE silver.orders USING DELTA LOCATION '{orders_path}'")

# Save Dispatch data as Delta and register table
silver_dispatch.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(dispatch_path)
spark.sql("DROP TABLE IF EXISTS silver.dispatch")
spark.sql(f"CREATE TABLE silver.dispatch USING DELTA LOCATION '{dispatch_path}'")

# Save Inventory data as Delta and register table
silver_inventory.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(inventory_path)
spark.sql("DROP TABLE IF EXISTS silver.inventory")
silver_inventory.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(inventory_path)
spark.sql(f"CREATE TABLE silver.inventory USING DELTA LOCATION '{inventory_path}'")

# Save Sales data as Delta and register table
silver_sales.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(sales_path)
spark.sql("DROP TABLE IF EXISTS silver.sales")
spark.sql(f"CREATE TABLE silver.sales USING DELTA LOCATION '{sales_path}'")

print("✅ Silver Delta tables created and registered!")
spark.sql("SHOW TABLES IN silver").show()

✅ Silver Delta tables created and registered!
+---------+-----------+-----------+
|namespace|  tableName|isTemporary|
+---------+-----------+-----------+
|   silver|   dispatch|      false|
|   silver|  inventory|      false|
|   silver|     orders|      false|
|   silver|      sales|      false|
|         | v_dispatch|       true|
|         |v_inventory|       true|
|         |   v_orders|       true|
|         |   v_routes|       true|
|         |    v_sales|       true|
+---------+-----------+-----------+



#Upsert Code to update data on daily basis or if there is no match create new row or column as per new data

In [265]:
new_orders.createOrReplaceTempView("incoming_orders")

spark.sql("""
    MERGE INTO silver.orders AS target
    USING incoming_orders AS source
    ON target.order_id = source.order_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

spark.sql("SELECT * FROM silver.orders").createOrReplaceTempView("v_orders")

print("✅ UPSERT done!")
spark.sql("SELECT * FROM v_orders ORDER BY order_id").show()

✅ UPSERT done!
+--------+----------+---------+----------+--------+-----+------------+----------+
|order_id|order_date|dealer_id|product_id|quantity|price|order_status|    region|
+--------+----------+---------+----------+--------+-----+------------+----------+
|       1|2026-03-01|      101|      1001|      50|  300|   delivered|     North|
|       2|2026-03-02|      102|      1002|      35|  320|   completed|North_West|
|       3|2026-03-03|      103|      1001|      20|  300|   cancelled|     South|
|       4|2026-03-04|      101|      1003|      40|  350|   completed|      East|
|       5|2026-03-05|      104|      1002|      60|  310|      placed|     North|
|       7|2026-03-07|      106|      1005|      25|    0|   delivered|     South|
|       8|2026-03-08|      101|      1001|      50|  300|   delivered|     North|
|      10|2026-03-10|      108|      1007|      45|  360|   cancelled|   unknown|
|      11|2026-03-11|      109|      1008|      55|  380|      placed|North_east|
+

## 3. Gold Layer: Business Analytics & SQL

### 3.1 Registering Temp Views for SQL Analysis

In [254]:
spark.sql("SELECT * FROM silver.orders").createOrReplaceTempView("v_orders")
spark.sql("SELECT * FROM silver.dispatch").createOrReplaceTempView("v_dispatch")
spark.sql("SELECT * FROM silver.inventory").createOrReplaceTempView("v_inventory")
spark.sql("SELECT * FROM silver.sales").createOrReplaceTempView("v_sales")
routes_df.createOrReplaceTempView("v_routes")


In [255]:
spark.sql("SELECT * FROM v_sales").show()

+---------+----------+----------+-------------+-------+------+
|dealer_id|product_id| sale_date|quantity_sold|revenue|region|
+---------+----------+----------+-------------+-------+------+
|      101|      1001|2026-03-06|           45|  13500| North|
|      102|      1002|2026-03-06|           25|   8000|  West|
|      103|      1001|2026-03-07|           15|   4500| South|
|      101|      1003|2026-03-07|           35|  12250|  East|
+---------+----------+----------+-------------+-------+------+



##1:Best route (less fuel used with minimum time)

In [256]:
best_routes = spark.sql("""
SELECT
    route,
    AVG(time_taken_min) as avg_time,
    AVG(fuel_used_litre) as avg_fuel
FROM v_routes
GROUP BY route
ORDER BY avg_time ASC, avg_fuel ASC
LIMIT 3
""")

best_routes.show()

+-----+--------+------------------+
|route|avg_time|          avg_fuel|
+-----+--------+------------------+
|    C|    26.0|               4.1|
|    A|    31.0|5.1000000000000005|
|    B|    40.0|               6.5|
+-----+--------+------------------+



###  2: Dealer-wise Sales & Revenue Summary

In [257]:
dealer_performance = spark.sql("""
    SELECT
        dealer_id,
        SUM(revenue) as total_revenue
    FROM v_sales
    GROUP BY dealer_id
""")

print("Total revenue generated by per dealer given below:-")
dealer_performance.show()

Total revenue generated by per dealer given below:-
+---------+-------------+
|dealer_id|total_revenue|
+---------+-------------+
|      101|        25750|
|      102|         8000|
|      103|         4500|
+---------+-------------+



##3: Inventory Critical Alerts

In [258]:
gold_inventory_alerts = spark.sql("""
    SELECT
        product_id,
        warehouse_id,
        stock_quantity,
        reorder_level,
        CASE
            WHEN stock_quantity <= reorder_level THEN '🔴 CRITICAL'
            WHEN stock_quantity <= reorder_level * 1.5 THEN '🟡 LOW'
            ELSE '🟢 OK'
        END AS stock_status
    FROM v_inventory
    ORDER BY stock_quantity ASC
""")

print("═══ Inventory Alerts ═══")
gold_inventory_alerts.show()

═══ Inventory Alerts ═══
+----------+------------+--------------+-------------+------------+
|product_id|warehouse_id|stock_quantity|reorder_level|stock_status|
+----------+------------+--------------+-------------+------------+
|      1004|          W2|            45|           60| 🔴 CRITICAL|
|      1003|          W2|           200|           50|       🟢 OK|
|      1002|          W1|           300|           80|       🟢 OK|
|      1001|          W1|           500|          100|       🟢 OK|
+----------+------------+--------------+-------------+------------+



### 4: Order vs Dispatch Fulfillment Leakage Analysis

In [261]:
gold_fulfillment = spark.sql("""
    SELECT
        o.order_id,
        o.quantity as ordered_qty,
        COALESCE(d.quantity_dispatched, 0) as dispatched_qty
  --using left join on order we here we can sell all order if there is no dispatch status
    FROM v_orders o
    LEFT JOIN v_dispatch d
    ON o.order_id = d.order_id
""")

gold_fulfillment.show()

+--------+-----------+--------------+
|order_id|ordered_qty|dispatched_qty|
+--------+-----------+--------------+
|       1|         50|            50|
|       2|         30|            30|
|       3|         20|            20|
|       4|         40|            35|
|       5|         60|            60|
|       7|         25|            55|
|       8|         50|             0|
|      10|         45|             0|
+--------+-----------+--------------+



### 5 Saving Final Gold Tables to Delta

In [270]:
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

# Define paths for Gold tables
dealer_kpi_path = "/content/spark-warehouse/gold/dealer_kpi"
inventory_alerts_path = "/content/spark-warehouse/gold/inventory_alerts"
fulfillment_leakage_path = "/content/spark-warehouse/gold/fulfillment_leakage"
route_path = "/content/spark-warehouse/gold/route_optimization"

#route for route optimization
best_routes.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(route_path)
spark.sql("DROP TABLE IF EXISTS gold.route_optimization")

spark.sql(f"""
CREATE TABLE gold.route_optimization
USING DELTA
LOCATION '{route_path}'
""")

# Save gold_dealer_performance as Delta and register table
gold_dealer_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(dealer_kpi_path)
spark.sql("DROP TABLE IF EXISTS gold.dealer_kpi")
spark.sql(f"CREATE TABLE gold.dealer_kpi USING DELTA LOCATION '{dealer_kpi_path}'")

# Save gold_inventory_alerts as Delta and register table
gold_inventory_alerts.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(inventory_alerts_path)
spark.sql("DROP TABLE IF EXISTS gold.inventory_alerts")
spark.sql(f"CREATE TABLE gold.inventory_alerts USING DELTA LOCATION '{inventory_alerts_path}'")

# Save gold_fulfillment_leakage as Delta and register table
gold_fulfillment_leakage.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(fulfillment_leakage_path)
spark.sql("DROP TABLE IF EXISTS gold.fulfillment_leakage")
spark.sql(f"CREATE TABLE gold.fulfillment_leakage USING DELTA LOCATION '{fulfillment_leakage_path}'")

print("✅ All Gold tables saved successfully!")

✅ All Gold tables saved successfully!


In [271]:
print("📊 Gold Tables Created:\n")

spark.sql("SHOW TABLES IN gold").show()

print("Best Route")
spark.sql("SELECT * FROM gold.route_optimization").show()

print("📈 Dealer KPI:")
spark.sql("SELECT * FROM gold.dealer_kpi").show()

print("📦 Inventory Alerts:")
spark.sql("SELECT * FROM gold.inventory_alerts").show()

print("🚚 Fulfillment Leakage:")
spark.sql("SELECT * FROM gold.fulfillment_leakage").show()

📊 Gold Tables Created:

+---------+-------------------+-----------+
|namespace|          tableName|isTemporary|
+---------+-------------------+-----------+
|     gold|         dealer_kpi|      false|
|     gold|fulfillment_leakage|      false|
|     gold|   inventory_alerts|      false|
|     gold| route_optimization|      false|
|         |    incoming_orders|       true|
|         |         v_dispatch|       true|
|         |        v_inventory|       true|
|         |           v_orders|       true|
|         |           v_routes|       true|
|         |            v_sales|       true|
+---------+-------------------+-----------+

Best Route
+-----+--------+------------------+
|route|avg_time|          avg_fuel|
+-----+--------+------------------+
|    C|    26.0|               4.1|
|    A|    31.0|5.1000000000000005|
|    B|    40.0|               6.5|
+-----+--------+------------------+

📈 Dealer KPI:
+---------+-------------+
|dealer_id|total_revenue|
+---------+-------------+
|  

##🔹 Project Conclusion
Built a scalable data pipeline using the Bronze–Silver–Gold architecture to process and analyze cement sales and logistics data

Used hardcoded sample data to ensure easy testing and reproducibility across environments

Implemented Delta Lake for reliable storage and efficient data handling

Applied UPSERT (MERGE) operations to support incremental data loading, eliminating the need to recreate tables or reprocess entire datasets

Performed analysis using SQL and PySpark, including:
Order vs Dispatch fulfillment and leakage
Route optimization based on time and fuel efficiency
Revenue analysis per dealer for business insights
Designed the Gold layer to deliver business-ready KPIs for decision-making